# 04 – Baseline: BoW / TF-IDF × Naive Bayes / Logistic Regression (Mốc 1)

Mỗi mô hình báo **hai con số**: F1-macro trên ViSD4SA test (trong miền của A) và F1-macro trên tập vàng Shopee.

⚠️ Nếu chưa có `labeling/gold_500.csv`, notebook **tạm** chấm trên cả kho vàng (746 câu, 10 sản phẩm) bằng **nhãn sao**. Con số đó chỉ để tham khảo, không phải kết quả cuối. Gán nhãn xong thì chạy lại toàn bộ notebook.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report

P = "../data/processed"
A = pd.read_csv(f"{P}/visd_train.csv")[["text_prep", "label"]]
A_test = pd.read_csv(f"{P}/visd_test.csv")[["text_prep", "label"]]
B = pd.read_csv(f"{P}/shopee_train.csv").rename(columns={"weak_label": "label"})[["text_prep", "label"]]
Aug = pd.read_csv(f"{P}/aug_train.csv")[["text_prep", "weak_label"]].rename(columns={"weak_label": "label"})   # file đã có cột label gốc (positive/negative)
pool = pd.read_csv(f"{P}/shopee_gold_pool.csv")

GOLD = "../labeling/gold_500.csv"
if os.path.exists(GOLD):
    g = pd.read_csv(GOLD)[["id", "sentiment"]].dropna()
    g["sentiment"] = g.sentiment.str.strip().str.lower()
    gold = g[g.sentiment.isin(["pos", "neg", "neu"])].merge(pool[["id", "text_prep"]], on="id")
    GOLD_NAME = f"tập vàng (nhãn tay, {len(gold)} câu)"
else:
    gold = pool.rename(columns={"weak_label": "sentiment"})[["text_prep", "sentiment"]]
    GOLD_NAME = f"TẠM: kho vàng nhãn sao ({len(gold)} câu)"
print("Đánh giá ngoài miền trên:", GOLD_NAME)
print(gold.sentiment.value_counts())

train_sets = {"A": A, "B": B, "A+B": pd.concat([A, B], ignore_index=True), "A+B+Aug": pd.concat([A, B, Aug], ignore_index=True)}
for k, d in train_sets.items():
    print(f"{k:8s}", len(d), dict(d.label.value_counts()))

Đánh giá ngoài miền trên: TẠM: kho vàng nhãn sao (746 câu)
sentiment
pos    536
neg    160
neu     50
Name: count, dtype: int64
A        7344 {'pos': np.int64(4596), 'neg': np.int64(2440), 'neu': np.int64(308)}
B        1794 {'pos': np.int64(1165), 'neg': np.int64(437), 'neu': np.int64(192)}
A+B      9138 {'pos': np.int64(5761), 'neg': np.int64(2877), 'neu': np.int64(500)}
A+B+Aug  10343 {'pos': np.int64(6413), 'neg': np.int64(3430), 'neu': np.int64(500)}


In [2]:
vecs = {"BoW": lambda: CountVectorizer(ngram_range=(1, 2), min_df=2),
        "TF-IDF": lambda: TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)}
clfs = {"NB": lambda: MultinomialNB(),
        "LR": lambda: LogisticRegression(max_iter=2000, class_weight="balanced")}

rows, models = [], {}
for cfg, d in train_sets.items():
    for vn, v in vecs.items():
        for cn, c in clfs.items():
            m = make_pipeline(v(), c()).fit(d.text_prep, d.label)
            models[(cfg, f"{vn}+{cn}")] = m
            rows.append({"train": cfg, "model": f"{vn}+{cn}",
                         "F1_ViSD_test": f1_score(A_test.label, m.predict(A_test.text_prep), average="macro"),
                         "F1_gold": f1_score(gold.sentiment, m.predict(gold.text_prep), average="macro")})
res = pd.DataFrame(rows)
res.to_csv("../report/baseline_results.csv", index=False)
print("Ngoài miền:", GOLD_NAME)
display(res.pivot(index="model", columns="train", values="F1_gold").round(3))
print("Trong miền: ViSD4SA test")
display(res.pivot(index="model", columns="train", values="F1_ViSD_test").round(3))

Ngoài miền: TẠM: kho vàng nhãn sao (746 câu)


train,A,A+B,A+B+Aug,B
model,,,,
BoW+LR,0.492,0.611,0.625,0.646
BoW+NB,0.543,0.563,0.544,0.599
TF-IDF+LR,0.534,0.611,0.604,0.654
TF-IDF+NB,0.448,0.488,0.477,0.511


Trong miền: ViSD4SA test


train,A,A+B,A+B+Aug,B
model,,,,
BoW+LR,0.739,0.725,0.725,0.504
BoW+NB,0.586,0.595,0.576,0.522
TF-IDF+LR,0.712,0.714,0.715,0.496
TF-IDF+NB,0.543,0.545,0.527,0.493


In [3]:
# Báo cáo chi tiết từng lớp cho mô hình tốt nhất trên tập vàng
best = res.loc[res.F1_gold.idxmax()]
print("Tốt nhất:", best.train, best.model, f"F1_gold = {best.F1_gold:.3f}")
m = models[(best.train, best.model)]
print(classification_report(gold.sentiment, m.predict(gold.text_prep), digits=3))

Tốt nhất: B TF-IDF+LR F1_gold = 0.654


              precision    recall  f1-score   support

         neg      0.713     0.713     0.713       160
         neu      0.247     0.400     0.305        50
         pos      0.974     0.918     0.945       536

    accuracy                          0.839       746
   macro avg      0.645     0.677     0.654       746
weighted avg      0.869     0.839     0.852       746



### Tự kiểm tra: mô hình có học “lối tắt” không?

In 20 từ có trọng số lớn nhất của từng lớp (TF-IDF+LR). Nếu có tên sản phẩm hoặc thương hiệu thì đó là dấu hiệu mô hình học lối tắt.

In [4]:
for cfg in ["B", "A+B"]:
    pipe = models[(cfg, "TF-IDF+LR")]
    vocab = np.array(pipe[0].get_feature_names_out())
    lr = pipe[1]
    print(f"===== train = {cfg} =====")
    for i, cls in enumerate(lr.classes_):
        top = vocab[np.argsort(lr.coef_[i])[::-1][:20]]
        print(f"{cls}: {', '.join(top)}")
    print()

===== train = B =====
neg: tệ, kém, không, thất_vọng, rách, hỏng, lỗi, làm_ăn, như, quần, sạc, không nên, trả_lời, mà, quá tệ, xấu, quá, đặt, cửa_hàng giao, không đúng
neu: chỉ, mỏng, có, thì, hơi, khăn, mà không, vcl, nhưng, tạm, không như, 38, nước, hay, quá lâu, tiền nào, mỏng quá, trắng không, trong, em
pos: tốt, ổn, đẹp, đúng, nhanh, sản_phẩm, rất, giao hàng, giá, rẻ, đóng_gói, hàng, chất_lượng, và, dùng, nên, hàng nhanh, ủng_hộ, cẩn_thận, mùi

===== train = A+B =====
neg: tệ, không, thất_vọng, kém, chán, lỗi, đơ, thua, không nên, chậm, mà, quá tệ, điện_thoại, đổi, tắt, mất, yếu, bị, cảm_ứng, màn_hình
neu: tạm, tạm ổn, bình_thường, cũng ổn, mỏng, tạm được, cũng, khăn, được, ổn nhưng, nhưng, chỉ, dùng cũng, hơi, dây, quá lâu, rớt, cửa_hàng, nước, cũng được
pos: ổn, tốt, rất, đẹp, tuyệt_vời, ngon, mượt, trâu, sản_phẩm, chê, mọi, không bị, mọi thứ, thứ, rẻ, đều ổn, đúng, đều, nhiệt_tình, pin trâu

